# Exercise 006

<a href="https://colab.research.google.com/github/FAIRChemistry/PythonProgramming2025/blob/master/exercises/Exercise006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Please execute this cell to download the necessary data
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/all_sequences.fasta

--2026-05-30 09:13:55--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/all_sequences.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2940315 (2.8M) [text/plain]
Saving to: ‘all_sequences.fasta’

all_sequences.fasta 100%[===================>]   2.80M  18.1MB/s    in 0.2s    

2026-05-30 09:13:56 (18.1 MB/s) - ‘all_sequences.fasta’ saved [2940315/2940315]



# DNASequence class

Read the FASTA file `all_sequences.fasta` and store header info and sequence in a suitable class. Make sure that at the initialization of the object, the following atrributes are present:

* `id`
* `organism`
* `sequence`
* `gc_content`
* `length`

**Tips**

> * Your `__init__`-method arguments do not have to contain all expected attributes if you can derive them from another attribute. The `__init__`-method is a function and you can execute any code you want upon initialization. Make sure to assign your calculation to the appropriate attribute via `self.xyz`.
> * [Dataclasses](https://docs.python.org/3/library/dataclasses.html) are a convinient way to create classes that simply hold data. You can make use of them to simplify the process due to the automatic generation of a `__init__`-method. But keep in mind that this excludes additional calculation you would have otherwise put into your custom `__init__`-method.

In [2]:
class DNASequence:
    def __init__(self, header: str, sequence: str):
        """
        Initializes a DNASequence object.

        Args:
            header (str): The FASTA header line (e.g., ">Sequence_1 [organism=Homo sapiens]")
            sequence (str): The actual nucleotide sequence string
        """
        # Parse the header to extract id and organism
        # Assuming format like: >id [organism=Name] or >id organism_name
        # We strip the leading '>' from the header first
        header_clean = header.strip().lstrip('>')

        # Example extraction logic:
        if "[organism=" in header_clean:
            self.id = header_clean.split(" [")[0]
            self.organism = header_clean.split("[organism=")[1].replace("]", "")
        else:
            # Fallback if the format varies
            parts = header_clean.split(maxsplit=1)
            self.id = parts[0]
            self.organism = parts[1] if len(parts) > 1 else "Unknown"

        self.sequence = sequence.strip().upper()
        self.length = len(self.sequence)

        # Calculate GC content: (G + C) / total length
        g_count = self.sequence.count('G')
        c_count = self.sequence.count('C')
        self.gc_content = (g_count + c_count) / self.length if self.length > 0 else 0.0

    def __repr__(self):
        return f"DNASequence(id='{self.id}', organism='{self.organism}', length={self.length}, gc_content={self.gc_content:.2%})"

## Magic Methods - Alignment by `==`

Can you extend the class to output the identity between the two sequences (stored as an attribute) when the `==` comparison operator is used? Apply the implementation to two sequences that you have chosen and use the supplied `get_identity` function.

Learn more about [Magic methods](https://realpython.com/python-magic-methods/)

In [3]:
# Execute this cell to install all necessary packages
%pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.5 MB/s eta 0:00:00


In [5]:
# Execute this cell to use the alignment function
from Bio import pairwise2


def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython

    Args:
        seq1 (str): Query sequence to align to
        seq2 (str): Target sequence to align with

    Returns:
        float: Identity of the resulting alignment

    """
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)

In [6]:
# Ensure biopython is available and get_identity is defined
from Bio import pairwise2

def get_identity(seq1: str, seq2: str):
    """Aligns two sequences using BioPython"""
    return pairwise2.align.globalxx(seq1, seq2, score_only=True) / len(seq1)


class AlignableDNASequence(DNASequence):
    """Extended DNASequence class implementing the == operator for alignment."""

    def __eq__(self, other):
        if not isinstance(other, AlignableDNASequence):
            return NotImplemented

        # Calculate alignment identity using the provided function
        identity = get_identity(self.sequence, other.sequence)
        return identity

In [7]:
# Let's create two mock sequences to test the alignment comparison
seq_a = AlignableDNASequence(">Seq1 [organism=Species A]", "ATCGATCGATCG")
seq_b = AlignableDNASequence(">Seq2 [organism=Species B]", "ATCGATCGATTT")

# Trigger the alignment by comparing them with ==
alignment_identity = seq_a == seq_b

print(f"Sequence A: {seq_a.sequence}")
print(f"Sequence B: {seq_b.sequence}")
print(f"Alignment Identity via '==': {alignment_identity:.2%}")

Sequence A: ATCGATCGATCG
Sequence B: ATCGATCGATTT
Alignment Identity via '==': 83.33%
